# LangChain - create_agent로 간단한 에이전트 만들기

LangChain 1.0의 `create_agent` 함수를 사용하여 프로덕션 수준의 에이전트를 쉽게 만들 수 있습니다.

## 학습 목표
- `create_agent` 함수의 핵심 파라미터 이해
- 다양한 설정으로 에이전트 커스터마이징
- ReAct 패턴의 작동 원리 이해

## 1. 환경 설정

In [1]:
import os 
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

print(f"OPENAI_API_KEY: {'설정됨' if api_key else '미설정'}")

OPENAI_API_KEY: 설정됨


## 2. create_agent 기본 사용법

`create_agent`는 LangGraph 기반의 에이전트를 생성합니다.

### 핵심 파라미터
- `model`: 사용할 LLM (문자열 또는 모델 인스턴스)
- `tools`: 에이전트가 사용할 도구 리스트
- `system_prompt`: 에이전트의 행동을 안내하는 시스템 프롬프트

- tool 1개 선언

In [2]:
# 1. 라이브러리 import 
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_agent

# 2. 도구 정의 
@tool
def get_weather(city: str) -> str:
    """도시의 날씨를 알려주는 함수"""
    return f"{city}의 날씨는 맑음입니다."

# 3. 모델 생성 
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 4. agent 생성 
agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt="질문에 맞는 도구를 골라 사용하세요."
)

# 5. 실행 
response = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "서울 날씨 알려줘"}
        ]
    }
)

# 6. 마지막 메시지 출력 
print(response["messages"][-1].content)


서울의 날씨는 맑습니다.


- tool 2개 선언

In [5]:
# pip install langchain-community langchain-openai
# pip install wikipedia numexpr
!pip install -U ddgs

  Using cached ddgs-9.14.4-py3-none-any.whl.metadata (20 kB)
  Using cached primp-1.3.1-cp310-abi3-win_amd64.whl.metadata (3.8 kB)
  Using cached lxml-6.1.1-cp312-cp312-win_amd64.whl.metadata (3.6 kB)
  Using cached fake_useragent-2.2.0-py3-none-any.whl.metadata (17 kB)
  Using cached brotli-1.2.0-cp312-cp312-win_amd64.whl.metadata (6.3 kB)
  Using cached socksio-1.0.0-py3-none-any.whl.metadata (6.1 kB)
Using cached ddgs-9.14.4-py3-none-any.whl (70 kB)
Using cached fake_useragent-2.2.0-py3-none-any.whl (161 kB)
Using cached socksio-1.0.0-py3-none-any.whl (12 kB)
Using cached lxml-6.1.1-cp312-cp312-win_amd64.whl (4.0 MB)
Using cached primp-1.3.1-cp310-abi3-win_amd64.whl (4.7 MB)
Using cached brotli-1.2.0-cp312-cp312-win_amd64.whl (369 kB)

   ------------- -------------------------- 2/6 [primp]
   -------------------- ------------------- 3/6 [lxml]
   -------------------- ------------------- 3/6 [lxml]
   -------------------- ------------------- 3/6 [lxml]
   -------------------- ------


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# 1. 필요한 모듈 임포트 및 환경 변수 설정 
from dotenv import load_dotenv
import os

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

load_dotenv()

# 2. LLM 초기화 
client = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

# 3. 도구 정의
# 3-1. 검색 도구
search_tool = DuckDuckGoSearchRun()

# 3-2. 계산 도구
@tool
def calculator(expression: str) -> str:
    """수학 계산이 필요할 때 사용하세요. 예: 23*7, (15+3)/2"""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"계산 중 오류가 발생했습니다: {e}"

tools = [search_tool, calculator]

# 4. Agent 생성 
agent = create_agent(
    model=client,
    tools=tools,
    system_prompt=(
        "당신은 질문에 답하는 도우미입니다. "
        "인물, 개념, 사건 등에 대한 정보 검색이 필요하면 검색 도구를 사용하고, "
        "계산이 필요하면 calculator 도구를 사용하세요."
    )
)

# 5. 메세지 기반 실행 함수 정의
def ask_agent(user_question: str):
    result = agent.invoke(
        {
            "messages": [
                {"role": "user", "content": user_question}
            ]
        }
    )
    return result

In [ ]:
# 6. 테스트 1: 검색
result = ask_agent("Jeff Bezos wikipedia")

print(result)

print("\n=== 최종 답변 ===")
print(result["messages"][-1].content)

{'messages': [HumanMessage(content='Jeff Bezos wikipedia', additional_kwargs={}, response_metadata={}, id='f323267c-f26f-4f4d-be64-e17124044f8c'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 157, 'total_tokens': 176, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_3d805c4100', 'id': 'chatcmpl-E7NEtDnBLBIsT9NeQDNGGmZaIyl9Z', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fb3b3-d30a-7001-b898-26992ea6fd9f-0', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'Jeff Bezos Wikipedia'}, 'id': 'call_EfUCGvnjdKh1LIPRq1inqNVi', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens'

In [8]:
# 6. 테스트 2:단순 계산 수행
result = ask_agent("23 * 7은 얼마야?")
print(result["messages"][-1].content)
# → Calculator 도구를 이용해 계산 수행

23 * 7은 161입니다.


In [9]:
# 6. 테스트 3 : 정보 추론 + 계산 결합
result = ask_agent(
    "제프 베조스가 1964년생이면 2024년 기준 나이는 몇 살인가?"
)
print(result["messages"][-1].content)

for msg in result["messages"]:
    print("\n====================")
    print("message type:", type(msg).__name__)
    print("content:", getattr(msg, "content", ""))

    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print("tool_calls:", msg.tool_calls)

    if type(msg).__name__ == "ToolMessage":
        print("tool name:", msg.name)
        print("tool result:", msg.content)

제프 베조스는 2024년 기준으로 60세입니다.

message type: HumanMessage
content: 제프 베조스가 1964년생이면 2024년 기준 나이는 몇 살인가?

message type: AIMessage
content: 
tool_calls: [{'name': 'calculator', 'args': {'expression': '2024 - 1964'}, 'id': 'call_84FOvgTjHG6BPniIRhxxtjco', 'type': 'tool_call'}]

message type: ToolMessage
content: 60
tool name: calculator
tool result: 60

message type: AIMessage
content: 제프 베조스는 2024년 기준으로 60세입니다.


## 3. ReAct 패턴 이해하기

에이전트는 **ReAct** (Reasoning + Acting) 패턴을 따릅니다:
1. **생각(Reasoning)**: 무엇을 해야 할지 결정
2. **행동(Acting)**: 도구 호출
3. **관찰(Observation)**: 결과 확인
4. 필요시 1-3 반복


- hasattr() : **“이 객체에 이런 속성(기능/데이터)이 있는지 확인하는 함수”**

In [14]:
# 정보 추론 + 계산 결합
result = ask_agent("2026년 현재 대한민국 대통령이 누구야? 그리고 2030년 기준으로 대통령의 나이 계산해줘.")
print(result["messages"][-1].content)


# for msg in result["messages"]:
#     print("\n====================")
#     print("message type:", type(msg).__name__)
#     print("content:", getattr(msg, "content", ""))

#     if hasattr(msg, "tool_calls") and msg.tool_calls:
#         print("tool_calls:", msg.tool_calls)

#     if type(msg).__name__ == "ToolMessage":
#         print("tool name:", msg.name)
#         print("tool result:", msg.content)
        
for i, msg in enumerate(result["messages"]):
    msg_type = msg.__class__.__name__

    # Agent가 Tool 호출을 결정한 메시지
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        tool_names = [tool_call["name"] for tool_call in msg.tool_calls]
        print(f"[{i}] {msg_type}: 도구 호출 - {tool_names}")

    # Tool 실행 결과 메시지
    elif msg_type == "ToolMessage":
        content = msg.content if isinstance(msg.content, str) else str(msg.content)
        print(f"[{i}] {msg_type} (도구: {msg.name}): {content[:100]}...")

    # 일반 메시지
    else:
        content = msg.content if isinstance(msg.content, str) else str(msg.content)
        print(
            f"[{i}] {msg_type}: {content[:100]}..."
            if len(content) > 100
            else f"[{i}] {msg_type}: {content}"
        )

2026년 현재 대한민국 대통령은 이재명입니다. 2030년 기준으로 이재명의 나이는 62세입니다.
[0] HumanMessage: 2026년 현재 대한민국 대통령이 누구야? 그리고 2030년 기준으로 대통령의 나이 계산해줘.
[1] AIMessage: 도구 호출 - ['duckduckgo_search']
[2] ToolMessage (도구: duckduckgo_search): 이재명 대통령, 2026년 신년사 발표 (서울=연합뉴스) 이재명 대통령이 청와대에서 2026년 신년사를 발표하고 있다. 이 대통령은 "올 한 해를 붉은 말처럼 힘차게 달리는 해로 ...
[3] AIMessage: 도구 호출 - ['calculator']
[4] ToolMessage (도구: calculator): 62...
[5] AIMessage: 2026년 현재 대한민국 대통령은 이재명입니다. 2030년 기준으로 이재명의 나이는 62세입니다.


## 4. 시스템 프롬프트 설계

좋은 시스템 프롬프트는 에이전트의 행동을 명확하게 안내합니다.

In [ ]:
# 역할 기반 시스템 프롬프트
CUSTOMER_SERVICE_PROMPT = """당신은 온라인 쇼핑몰의 고객 서비스 담당자입니다.

<역할>
- 고객 문의에 친절하고 전문적으로 응답
- 제품, 배송, 반품 관련 질문 처리
- 필요시 도구를 사용하여 정보 조회
</역할>

<톤앤매너>
- 항상 존댓말 사용
- 긍정적이고 해결 지향적
- 명확하고 간결한 답변
</톤앤매너>

<제한사항>
- 개인정보 요청 금지
- 회사 정책 외의 약속 금지
</제안사항>
"""

@tool
def check_order_status(order_id: str) -> str:
    """주문 상태를 확인합니다."""
    orders = {
        "A12345": "배송 중 (내일 도착 예정)",
        "B67890": "상품 준비 중",
        "C11111": "배송 완료"
    }
    return orders.get(order_id, f"주문번호 {order_id}를 찾을 수 없습니다.")

@tool
def get_product_info(product_name: str) -> str:
    """제품 정보를 조회합니다."""
    products = {
        "무선 이어폰": "가격: 89,000원, 재고: 있음, 배송: 1-2일",
        "노트북 거치대": "가격: 35,000원, 재고: 품절, 입고예정: 다음주"
    }
    return products.get(product_name, f"{product_name} 정보를 찾을 수 없습니다.")

cs_agent = create_agent(
    model="gpt-4.1-mini",
    tools=[check_order_status, get_product_info],
    system_prompt=CUSTOMER_SERVICE_PROMPT
)

print("고객 서비스 에이전트가 생성되었습니다.")

고객 서비스 에이전트가 생성되었습니다.


In [16]:
# 고객 서비스 테스트
cs_queries = [
    "주문번호 A12345 배송 상태 확인해주세요",
    "무선 이어폰 재고 있나요?",
    "반품하고 싶은데 어떻게 하나요?"
]

# invoke 방식으로 고객 문의 처리
for query in cs_queries:
    result = cs_agent.invoke({"messages": [{"role": "user", "content": query}]})
    print(f"고객: {query}")
    print(f"상담원: {result['messages'][-1].content}")
    print("=" * 60)

고객: 주문번호 A12345 배송 상태 확인해주세요
상담원: 주문번호 A12345의 배송 상태는 현재 배송 중이며, 내일 도착 예정입니다. 다른 문의 사항 있으시면 말씀해 주세요.
고객: 무선 이어폰 재고 있나요?
상담원: 무선 이어폰은 현재 재고가 있습니다. 필요하시면 언제든지 주문 도와드리겠습니다. 추가로 궁금한 점 있으시면 말씀해 주세요.
고객: 반품하고 싶은데 어떻게 하나요?
상담원: 안녕하세요 고객님, 반품을 도와드리겠습니다.

저희 쇼핑몰의 반품 절차는 다음과 같습니다:
1. 받으신 상품과 함께 반품 요청을 접수해 주셔야 합니다.
2. 상품은 사용하지 않은 상태여야 하며, 포장 상태가 양호해야 합니다.
3. 반품 요청은 수령일로부터 7일 이내에 해주셔야 합니다.

고객님께서 주문하신 주문번호나 상품명을 알려주시면 더욱 정확한 안내를 드릴 수 있습니다. 부탁드릴게요!


In [17]:
result = cs_agent.invoke({
    "messages": [{"role": "user", "content": "주문번호 A12345 배송 상태 확인해주세요"}]
})

for i, msg in enumerate(result["messages"], 1):
    print(f"\n[{i}] {type(msg).__name__}")
    if hasattr(msg, "content"):
        print("content:", msg.content)

    # tool call 정보가 있으면 출력
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print("tool_calls:", msg.tool_calls)


[1] HumanMessage
content: 주문번호 A12345 배송 상태 확인해주세요

[2] AIMessage
content: 
tool_calls: [{'name': 'check_order_status', 'args': {'order_id': 'A12345'}, 'id': 'call_4dQHtO4XEAgG0stgA9aL0fkJ', 'type': 'tool_call'}]

[3] ToolMessage
content: 배송 중 (내일 도착 예정)

[4] AIMessage
content: 주문번호 A12345는 현재 배송 중이며, 내일 도착 예정입니다. 더 궁금한 점 있으시면 언제든 문의해 주세요.


In [18]:
for query in cs_queries:
    print(f"\n고객: {query}")

    for chunk in cs_agent.stream(
        {"messages": [{"role": "user", "content": query}]},
        stream_mode="updates",
        version="v2",
    ):
        if chunk["type"] != "updates":
            continue

        for step, data in chunk["data"].items():
            msg = data["messages"][-1]

            print(f"\n[STEP: {step}]")
            print(f"메시지 타입: {type(msg).__name__}")

            if hasattr(msg, "content_blocks"):
                print("content_blocks:", msg.content_blocks)
            elif hasattr(msg, "content"):
                print("content:", msg.content)

            if hasattr(msg, "tool_calls") and msg.tool_calls:
                print("tool_calls:", msg.tool_calls)

    print("\n" + "=" * 60)


고객: 주문번호 A12345 배송 상태 확인해주세요

[STEP: model]
메시지 타입: AIMessage
content_blocks: [{'type': 'tool_call', 'name': 'check_order_status', 'args': {'order_id': 'A12345'}, 'id': 'call_YHdeP1sd1W55Zqhztz6RNHNr'}]
tool_calls: [{'name': 'check_order_status', 'args': {'order_id': 'A12345'}, 'id': 'call_YHdeP1sd1W55Zqhztz6RNHNr', 'type': 'tool_call'}]

[STEP: tools]
메시지 타입: ToolMessage
content_blocks: [{'type': 'text', 'text': '배송 중 (내일 도착 예정)'}]

[STEP: model]
메시지 타입: AIMessage
content_blocks: [{'type': 'text', 'text': "주문번호 A12345의 배송 상태는 현재 '배송 중'이며, 내일 도착할 예정입니다. 다른 문의 사항 있으시면 언제든지 말씀해 주세요."}]


고객: 무선 이어폰 재고 있나요?

[STEP: model]
메시지 타입: AIMessage
content_blocks: [{'type': 'tool_call', 'name': 'get_product_info', 'args': {'product_name': '무선 이어폰'}, 'id': 'call_ysAlc9GzS8WkbPppdqjJ1I4m'}]
tool_calls: [{'name': 'get_product_info', 'args': {'product_name': '무선 이어폰'}, 'id': 'call_ysAlc9GzS8WkbPppdqjJ1I4m', 'type': 'tool_call'}]

[STEP: tools]
메시지 타입: ToolMessage
content_blocks: [{'type': 'text', 'te

## 정리

### 이 노트북에서 배운 것

1. **create_agent 기본 사용법**
   ```python
   agent = create_agent(
       model="gpt-4.1-mini",
       tools=[tool1, tool2],
       system_prompt="..."
   )
   ```

2. **ReAct 패턴**
   - 생각 → 행동 → 관찰 → 반복

3. **시스템 프롬프트 설계**
   - 역할, 톤앤매너, 제한사항 명시
   - XML 태그로 구조화 가능